In [ ]:
import re
import math
from collections import defaultdict, Counter

# -----------------------------
# DATASET
# -----------------------------
documents = [
    ("Free money now", "SPAM"),
    ("Hi mom how are you", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Lets catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50 off limited time", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes", "SPAM"),
    ("Can you send the report", "HAM"),
]

def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.split()

vocab = set()
class_word_counts = defaultdict(Counter)
class_doc_counts = defaultdict(int)
total_words_per_class = defaultdict(int)

for text, label in documents:
    class_doc_counts[label] += 1
    words = tokenize(text)
    vocab.update(words)
    
    for word in words:
        class_word_counts[label][word] += 1
        total_words_per_class[label] += 1

total_docs = len(documents)
V = len(vocab)

print("a. BAG OF WORDS (Word Frequency per Class)\n")

for label in class_word_counts:
    print(f"{label}:")
    for word, count in class_word_counts[label].items():
        print(f"   {word}: {count}")
    print()

print("\nb. PRIOR PROBABILITIES\n")

priors = {}
for label in class_doc_counts:
    priors[label] = class_doc_counts[label] / total_docs
    print(f"P({label}) = {class_doc_counts[label]}/{total_docs} = {priors[label]:.4f}")

print("\nc. LIKELIHOOD of each word given class (with Laplace smoothing)\n")

for label in class_word_counts:
    print(f"Likelihoods for class {label}:")
    for word in sorted(vocab):
        likelihood = (class_word_counts[label][word] + 1) / (total_words_per_class[label] + V)
        print(f"   P({word}|{label}) = {likelihood:.4f}")
    print()

def predict(sentence):
    words = tokenize(sentence)
    scores = {}

    for label in priors:
        score = math.log(priors[label])
        for word in words:
            likelihood = (class_word_counts[label][word] + 1) / (total_words_per_class[label] + V)
            score += math.log(likelihood)
        scores[label] = score

    return max(scores, key=scores.get)

print("\nd. CLASSIFICATION RESULTS\n")

test1 = "Limited offer, click here!"
test2 = "Meeting at 2 PM with the manager."

print("i. Limited offer, click here!")
print("Predicted Class:", predict(test1))
print()

print("ii. Meeting at 2 PM with the manager.")
print("Predicted Class:", predict(test2))


a. BAG OF WORDS (Word Frequency per Class)

SPAM:
   free: 2
   money: 1
   now: 1
   lowest: 1
   price: 1
   for: 2
   your: 1
   meds: 1
   win: 1
   a: 1
   iphone: 1
   today: 1
   get: 1
   50: 1
   off: 1
   limited: 1
   time: 1
   click: 1
   here: 1
   prizes: 1

HAM:
   hi: 1
   mom: 1
   how: 1
   are: 2
   you: 2
   we: 1
   still: 1
   on: 1
   for: 1
   dinner: 1
   lets: 1
   catch: 1
   up: 1
   tomorrow: 2
   at: 2
   the: 3
   office: 2
   meeting: 2
   3: 1
   pm: 1
   team: 1
   in: 1
   can: 1
   send: 1
   report: 1


b. PRIOR PROBABILITIES

P(SPAM) = 5/11 = 0.4545
P(HAM) = 6/11 = 0.5455

c. LIKELIHOOD of each word given class (with Laplace smoothing)

Likelihoods for class SPAM:
   P(3|SPAM) = 0.0152
   P(50|SPAM) = 0.0303
   P(a|SPAM) = 0.0303
   P(are|SPAM) = 0.0152
   P(at|SPAM) = 0.0152
   P(can|SPAM) = 0.0152
   P(catch|SPAM) = 0.0152
   P(click|SPAM) = 0.0303
   P(dinner|SPAM) = 0.0152
   P(for|SPAM) = 0.0455
   P(free|SPAM) = 0.0455
   P(get|SPAM) = 0.030

In [3]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB


documents = [
    "Free money now",
    "Hi mom how are you",
    "Lowest price for your meds",
    "Are we still on for dinner",
    "Win a free iPhone today",
    "Lets catch up tomorrow at the office",
    "Meeting at 3 PM tomorrow",
    "Get 50 off limited time",
    "Team meeting in the office",
    "Click here for prizes",
    "Can you send the report"
]

labels = [
    "SPAM", "HAM", "SPAM", "HAM", "SPAM",
    "HAM", "HAM", "SPAM", "HAM", "SPAM", "HAM"
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)

model = MultinomialNB()
model.fit(X, labels)

test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]

X_test = vectorizer.transform(test_sentences)
predictions = model.predict(X_test)

print("Using Scikit-Learn Multinomial Naïve Bayes\n")

print("i. Limited offer, click here!")
print("Predicted Class:", predictions[0])
print()

print("ii. Meeting at 2 PM with the manager.")
print("Predicted Class:", predictions[1])


Using Scikit-Learn Multinomial Naïve Bayes

i. Limited offer, click here!
Predicted Class: SPAM

ii. Meeting at 2 PM with the manager.
Predicted Class: HAM
